In [ ]:
# Install required libraries for the lab.
%pip install openai numpy

import os
import numpy as np
from openai import OpenAI

# Replace this with your real API key before running.
OPENAI_API_KEY = ""
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Phase 2: Matryoshka vs Binary Quantization

This phase compares three ways of representing embeddings: the original full-length vectors, truncated Matryoshka-style vectors, and binary-quantized vectors. The goal is to observe how much similarity quality is preserved while reducing memory usage.

## Step 1: Define Sentences

In [9]:
# Define five documents and one query on renewable energy.
documents = [
    "Solar panels convert sunlight into electricity for homes and businesses.",
    "Wind turbines generate power by capturing energy from moving air.",
    "Battery storage helps renewable energy systems provide power after sunset.",
    "Hydroelectric dams produce electricity by using flowing water to spin turbines.",
    "Geothermal plants use heat from beneath the Earth's surface to generate energy.",
]
query = "How can clean energy be stored and used when the sun is not shining?"

for index, document in enumerate(documents, start=1):
    print(f"Document {index}: {document}")

print(f"\nQuery: {query}")


Document 1: Solar panels convert sunlight into electricity for homes and businesses.
Document 2: Wind turbines generate power by capturing energy from moving air.
Document 3: Battery storage helps renewable energy systems provide power after sunset.
Document 4: Hydroelectric dams produce electricity by using flowing water to spin turbines.
Document 5: Geothermal plants use heat from beneath the Earth's surface to generate energy.

Query: How can clean energy be stored and used when the sun is not shining?


## Step 2: Generate Full Embeddings

In [10]:
# Generate full embeddings with the raw OpenAI SDK.
client = OpenAI(api_key=OPENAI_API_KEY)
all_texts = documents + [query]
embedding_response = client.embeddings.create(
    model="text-embedding-3-large",
    input=all_texts,
)

all_embeddings = [np.array(item.embedding, dtype=np.float32) for item in embedding_response.data]
document_embeddings_full = all_embeddings[:-1]
query_embedding_full = all_embeddings[-1]
full_dimension_count = len(query_embedding_full)

print(f"Full embedding dimensions: {full_dimension_count}")


Full embedding dimensions: 3072


## Step 3: Matryoshka Truncation

In [11]:
# Normalize truncated vectors manually and compare them with cosine similarity.
def l2_normalize(vector):
    denominator = np.sqrt(np.sum(vector ** 2))
    normalized = vector / denominator
    return normalized

def cosine_similarity_manual(vector_a, vector_b):
    numerator = np.dot(vector_a, vector_b)
    denominator = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    return numerator / denominator

document_embeddings_truncated = [embedding[:256] for embedding in document_embeddings_full]
query_embedding_truncated = query_embedding_full[:256]
document_embeddings_truncated_normalized = [l2_normalize(embedding) for embedding in document_embeddings_truncated]
query_embedding_truncated_normalized = l2_normalize(query_embedding_truncated)

truncated_cosine_scores = [
    cosine_similarity_manual(query_embedding_truncated_normalized, document_embedding)
    for document_embedding in document_embeddings_truncated_normalized
]

for index, score in enumerate(truncated_cosine_scores, start=1):
    print(f"Document {index} truncated cosine similarity: {score:.4f}")


Document 1 truncated cosine similarity: 0.4245
Document 2 truncated cosine similarity: 0.3122
Document 3 truncated cosine similarity: 0.5567
Document 4 truncated cosine similarity: 0.3005
Document 5 truncated cosine similarity: 0.3213


## Step 4: Binary Quantization

In [12]:
# Convert full vectors to binary and compare them with Hamming distance.
def to_binary(vector):
    return np.where(vector > 0, 1, 0)

def hamming_distance_manual(query_binary, doc_binary):
    hamming = np.sum(query_binary != doc_binary)
    return hamming

document_embeddings_binary = [to_binary(embedding) for embedding in document_embeddings_full]
query_embedding_binary = to_binary(query_embedding_full)
hamming_distances = [
    hamming_distance_manual(query_embedding_binary, document_embedding)
    for document_embedding in document_embeddings_binary
]

for index, distance in enumerate(hamming_distances, start=1):
    print(f"Document {index} Hamming distance: {distance}")


Document 1 Hamming distance: 1151
Document 2 Hamming distance: 1253
Document 3 Hamming distance: 915
Document 4 Hamming distance: 1312
Document 5 Hamming distance: 1216


## Step 5: Comparison Table

In [13]:
# Compare the full, truncated, and binary rankings side by side.
full_cosine_scores = [
    cosine_similarity_manual(query_embedding_full, document_embedding)
    for document_embedding in document_embeddings_full
]

full_ranking = np.argsort(-np.array(full_cosine_scores))
truncated_ranking = np.argsort(-np.array(truncated_cosine_scores))
binary_ranking = np.argsort(np.array(hamming_distances))

if np.array_equal(full_ranking, truncated_ranking) and np.array_equal(full_ranking, binary_ranking):
    ranking_note = "Both truncation and binary quantization preserved the full similarity ranking equally well."
elif np.array_equal(full_ranking, truncated_ranking):
    ranking_note = "Matryoshka truncation preserved the full-vector similarity ranking best."
elif np.array_equal(full_ranking, binary_ranking):
    ranking_note = "Binary quantization preserved the full-vector similarity ranking best."
else:
    truncated_rank_gap = np.sum(np.abs(full_ranking - truncated_ranking))
    binary_rank_gap = np.sum(np.abs(full_ranking - binary_ranking))
    ranking_note = "Matryoshka truncation preserved the full-vector similarity ranking best." if truncated_rank_gap <= binary_rank_gap else "Binary quantization preserved the full-vector similarity ranking best."

print("| Document | Full Cosine Score | Truncated Cosine Score | Hamming Distance |")
print("|---|---:|---:|---:|")
for index, document in enumerate(documents, start=1):
    print(
        f"| Document {index} | {full_cosine_scores[index - 1]:.4f} | {truncated_cosine_scores[index - 1]:.4f} | {hamming_distances[index - 1]} |"
    )

print(f"\nNote: {ranking_note}")


| Document | Full Cosine Score | Truncated Cosine Score | Hamming Distance |
|---|---:|---:|---:|
| Document 1 | 0.3960 | 0.4245 | 1151 |
| Document 2 | 0.2879 | 0.3122 | 1253 |
| Document 3 | 0.5875 | 0.5567 | 915 |
| Document 4 | 0.2483 | 0.3005 | 1312 |
| Document 5 | 0.3094 | 0.3213 | 1216 |

Note: Both truncation and binary quantization preserved the full similarity ranking equally well.


## Step 6: Memory Comparison

In [14]:
# Compare memory usage for full, truncated, and binary vectors.
full_dimensions = 3072
truncated_dimensions = 256
full_vector_bytes = full_dimensions * 4
truncated_vector_bytes = truncated_dimensions * 4
binary_vector_bytes = full_dimensions / 8
truncated_memory_saved_percent = ((full_vector_bytes - truncated_vector_bytes) / full_vector_bytes) * 100
binary_memory_saved_percent = ((full_vector_bytes - binary_vector_bytes) / full_vector_bytes) * 100

print(f"Full vector memory: {full_vector_bytes:.0f} bytes")
print(f"Truncated vector memory: {truncated_vector_bytes:.0f} bytes")
print(f"Binary vector memory: {binary_vector_bytes:.0f} bytes")
print(f"Truncated memory saved: {truncated_memory_saved_percent:.2f}%")
print(f"Binary memory saved: {binary_memory_saved_percent:.2f}%")


Full vector memory: 12288 bytes
Truncated vector memory: 1024 bytes
Binary vector memory: 384 bytes
Truncated memory saved: 91.67%
Binary memory saved: 96.88%


## Step 7: Reflection Questions

Q1: When reducing vector dimensions, what happens to magnitude and representation?

A1: When dimensions are reduced, part of the original signal is removed, so the vector no longer preserves the full representation exactly. Its magnitude also changes because fewer values are contributing to the overall length.



Q2: Why is L2 normalization required after truncation?

A2: Truncation changes the vector length, so without normalization the similarity score can be affected by scale instead of meaning. L2 normalization puts the truncated vectors back on a comparable scale so cosine similarity reflects direction more fairly.



Q3: How can binary quantization reduce memory footprint drastically while maintaining retrieval accuracy?

A3: Binary quantization stores only the sign pattern of each dimension instead of full floating-point values, which cuts memory usage dramatically. Even though some detail is lost, the overall structure of the embedding can still be strong enough to preserve useful nearest-neighbor relationships.



Q4: Which similarity metric should be used for binary vectors and why?

A4: Hamming distance is the right choice because binary vectors are made of bit positions, and Hamming distance directly counts how many of those positions differ. That makes it a simple and appropriate way to measure similarity after binary conversion.



